# 04 - Evaluate baselines and a chunking ablation

**Objective:** measure all 809 development claims against all 5,183 documents, inspect failures, and preserve a separate test protocol.
**Setup:** prepare SciFact first. No API keys or model downloads. Allow a few minutes depending on hardware.

We report Recall@k, MRR@k, and nDCG@k at k=1,5,10. Recall counts relevant documents recovered; MRR rewards the first relevant rank; nDCG accounts for ranked relevance. These are retrieval metrics, not factuality or answer-quality scores. The labels here are binary, though the nDCG implementation supports nonnegative graded labels.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rag_lab/core.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the Awesome-RAG checkout")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag_lab.core import load_dataset, chunk_document, LexicalIndex, assemble_context
corpus, queries, qrels, manifest = load_dataset(ROOT)
print(f"{len(corpus):,} abstracts; {len(qrels['train']):,} development claims; {len(qrels['test']):,} test claims")
print("Snapshot:", manifest["archive_sha256"])


## Evaluate fixed whole-document baselines

The BEIR `train` split is used for development in these notebooks. It is not presented as an unseen test set. Both lexical methods use the same corpus and tokenization. Inspectable metric definitions are in `rag_lab/core.py`; tests check hand-computed cases.


In [ ]:
from rag_lab.evaluation import evaluate, paired_bootstrap
from time import perf_counter
results = {}
for method in ["tfidf", "bm25"]:
    start = perf_counter()
    index = LexicalIndex(list(corpus.values()), method=method)
    results[method] = evaluate(index, queries, qrels["train"])
    print(method, results[method]["summary"], "total seconds:", round(perf_counter() - start, 2))


## Test chunking rather than assuming it helps

Split all abstracts into 120-word windows with 30-word overlap, and index each title with each chunk. Map hits back to parent documents and keep the highest-scoring chunk per document **before** truncating to k. This avoids counting overlapping chunks as separate relevant documents. It also changes term statistics, so this ablation measures the combined effect of chunking and reindexing.


In [ ]:
chunks = [chunk for doc in corpus.values() for chunk in chunk_document(doc, 120, 30)]
chunk_index = LexicalIndex(chunks, method="bm25")
results["bm25_chunks"] = evaluate(chunk_index, queries, qrels["train"])
print("Indexed chunks:", len(chunks))
print(results["bm25_chunks"]["summary"])


## Quantify variation across queries

A paired bootstrap resamples query IDs with replacement and reports a percentile interval for the mean nDCG@10 difference. It describes variation within these claims; correlated claims and incomplete judgments limit its interpretation. Latency is measured in-process on this machine and is not a service load test.


In [ ]:
print("BM25 minus TF-IDF:", paired_bootstrap(results["tfidf"], results["bm25"]))
print("Chunked minus whole BM25:", paired_bootstrap(results["bm25"], results["bm25_chunks"]))


## Read actual misses

Inspect both the retrieved titles and the judged evidence. Categorize vocabulary mismatch, document length, missing context, or ranking problems. Unjudged documents are scored as nonrelevant by these metrics but may contain useful evidence.


In [ ]:
misses = [row for row in results["bm25"]["per_query"] if row["metrics"]["recall@10"] < 1]
print("Development queries missing at least one relevant document in top 10:", len(misses))
for row in misses[:5]:
    qid = row["query_id"]
    print("\nCLAIM:", qid, queries[qid]["text"])
    print("JUDGED:", [(did, corpus[did]["title"]) for did in qrels["train"][qid]])
    print("TOP 3:", [(hit["document_id"], corpus[hit["document_id"]]["title"]) for hit in row["hits"][:3]])


## Separate test reporting

After fixing configurations on development data, run `python scripts/benchmark.py --split test` from the repository root. It evaluates both fixed whole-document baselines over all 300 test claims and writes `reports/scifact-test.json` with dataset hashes, configuration, source-code hashes, metrics, timings, and per-query rankings. The checked-in report is a reference run of those baselines, not a claim of state-of-the-art performance. Avoid repeatedly selecting parameters against it.

**Exercise:** extend the benchmark command with your development-selected chunk configuration and document the decision before evaluating it on test. For dense retrieval later, keep the same corpus, queries, and scoring protocol.

**Limitations:** this benchmark covers scientific abstracts and evidence retrieval. It does not validate OCR, long-document understanding, production access controls, generation quality, or unanswerability. Do not interpret a missing relevance label as an unanswerable claim.
